## 10. Summary and Next Steps

This notebook demonstrates:
- ✅ Loading YOLOv8 pre-trained models
- ✅ Configuring detection parameters
- ✅ Running inference on images
- ✅ Visualizing detection results
- ✅ Extracting detection details
- ✅ Real-time webcam detection
- ✅ Performance evaluation
- ✅ Batch processing

### Next Steps:
1. Try different model sizes (yolov8n, yolov8s, yolov8l, yolov8x)
2. Process videos using the functions in `utils/detector.py`
3. Fine-tune the model on custom datasets
4. Deploy to production with the `detect.py` script
5. Explore model training and validation

For more information, visit the [YOLOv8 Documentation](https://docs.ultralytics.com/)

In [ ]:
# Batch processing function
def batch_detect_images(image_paths, output_dir='./results'):
    """
    Process multiple images and save annotated versions
    
    Args:
        image_paths: List of image paths
        output_dir: Directory to save results
    """
    Path(output_dir).mkdir(exist_ok=True)
    results_summary = []
    
    print(f"Processing {len(image_paths)} images...")
    
    for idx, img_path in enumerate(image_paths, 1):
        if not Path(img_path).exists():
            print(f"  Skipped (not found): {img_path}")
            continue
        
        start = time.time()
        results = model(img_path, **detection_config, verbose=False)
        result = results[0]
        inference_time = time.time() - start
        
        # Save annotated image
        output_path = Path(output_dir) / f"detected_{Path(img_path).stem}.jpg"
        annotated = result.plot()
        cv2.imwrite(str(output_path), annotated)
        
        results_summary.append({
            'Image': Path(img_path).name,
            'Detections': len(result.boxes),
            'Time (ms)': f"{inference_time*1000:.1f}",
            'Output': str(output_path)
        })
        
        print(f"  [{idx}/{len(image_paths)}] {Path(img_path).name}: {len(result.boxes)} objects, {inference_time*1000:.1f}ms")
    
    return pd.DataFrame(results_summary)

# Example: Process the sample image
print("Batch processing example:")
batch_results = batch_detect_images(['./data/sample.jpg'])
print("\nBatch Processing Results:")
print(batch_results)

## 9. Batch Processing Multiple Images

In [ ]:
# Performance evaluation
def evaluate_model_performance(model, test_image_path, num_runs=5):
    """
    Evaluate model performance metrics
    
    Args:
        model: YOLO model
        test_image_path: Path to test image
        num_runs: Number of inference runs for averaging
    """
    inference_times = []
    all_detections = []
    
    print(f"Running performance evaluation ({num_runs} runs)...")
    
    for i in range(num_runs):
        start = time.time()
        results = model(test_image_path, **detection_config, verbose=False)
        inference_time = time.time() - start
        inference_times.append(inference_time)
        all_detections.append(len(results[0].boxes))
    
    # Calculate statistics
    avg_time = np.mean(inference_times)
    std_time = np.std(inference_times)
    min_time = np.min(inference_times)
    max_time = np.max(inference_times)
    
    print("\n--- Performance Metrics ---")
    print(f"Average Inference Time: {avg_time*1000:.2f} ms")
    print(f"Std Dev: {std_time*1000:.2f} ms")
    print(f"Min Time: {min_time*1000:.2f} ms")
    print(f"Max Time: {max_time*1000:.2f} ms")
    print(f"Average FPS: {1/avg_time:.2f}")
    print(f"Average Detections: {np.mean(all_detections):.1f}")
    
    return {
        'avg_time': avg_time,
        'std_time': std_time,
        'fps': 1/avg_time,
        'detections': np.mean(all_detections)
    }

# Run performance evaluation
performance = evaluate_model_performance(model, image_path, num_runs=5)

## 8. Performance Metrics

In [ ]:
# Real-time webcam detection function
def webcam_detection(duration=10, show_fps=True):
    """
    Real-time object detection from webcam
    
    Args:
        duration: Duration in seconds to run detection
        show_fps: Display FPS on the frame
    """
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Cannot open webcam")
        return
    
    start_time = time.time()
    frame_count = 0
    fps_list = []
    
    print(f"Starting webcam detection for {duration} seconds...")
    print("Press 'q' to quit early")
    
    while (time.time() - start_time) < duration:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Run detection
        frame_start = time.time()
        results = model(frame, **detection_config)
        result = results[0]
        
        # Get annotated frame
        annotated_frame = result.plot()
        
        # Calculate FPS
        frame_time = time.time() - frame_start
        fps = 1 / frame_time
        fps_list.append(fps)
        
        # Add FPS text if requested
        if show_fps:
            cv2.putText(annotated_frame, f"FPS: {fps:.1f}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(annotated_frame, f"Detections: {len(result.boxes)}", (10, 70),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display frame
        cv2.imshow('YOLOv8 Webcam Detection', annotated_frame)
        
        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        frame_count += 1
    
    cap.release()
    cv2.destroyAllWindows()
    
    # Print statistics
    if fps_list:
        print(f"\n--- Webcam Detection Statistics ---")
        print(f"Frames processed: {frame_count}")
        print(f"Average FPS: {np.mean(fps_list):.2f}")
        print(f"Min FPS: {min(fps_list):.2f}")
        print(f"Max FPS: {max(fps_list):.2f}")

# Uncomment to run webcam detection
# webcam_detection(duration=10)

## 7. Webcam Detection (Real-time)

In [ ]:
# Extract detection details
detections = []

for box in result.boxes:
    detection_info = {
        'Class': model.names[int(box.cls)],
        'Confidence': f"{float(box.conf):.4f}",
        'X1': f"{float(box.xyxy[0][0]):.1f}",
        'Y1': f"{float(box.xyxy[0][1]):.1f}",
        'X2': f"{float(box.xyxy[0][2]):.1f}",
        'Y2': f"{float(box.xyxy[0][3]):.1f}",
    }
    detections.append(detection_info)

# Create DataFrame for better visualization
df_detections = pd.DataFrame(detections)
print("\nDetection Results:")
print(df_detections.to_string(index=False))

# Summary statistics
print(f"\n--- Detection Summary ---")
print(f"Total detections: {len(detections)}")
if detections:
    class_counts = df_detections['Class'].value_counts()
    print("\nClass Distribution:")
    print(class_counts)

## 6. Extract and Display Detection Details

In [ ]:
# Visualize detections
def plot_detections(result, figsize=(12, 8)):
    """Plot detection results with bounding boxes and labels"""
    # Get annotated image
    im_array = result.plot()
    
    # Display
    plt.figure(figsize=figsize)
    plt.imshow(im_array[..., ::-1])  # Convert BGR to RGB
    plt.axis('off')
    plt.title(f'YOLOv8 Detection - {len(result.boxes)} objects detected')
    plt.tight_layout()
    plt.show()

# Plot results
plot_detections(result)

# Save annotated image
output_path = "./results/annotated_image.jpg"
Path(output_path).parent.mkdir(exist_ok=True)
annotated_image = result.plot()
cv2.imwrite(output_path, annotated_image)
print(f"Annotated image saved to {output_path}")

## 5. Display Detection Results

In [ ]:
# Perform detection on the sample image
image_path = "./data/sample.jpg"

print(f"Running inference on {image_path}...")
start_time = time.time()

results = model(image_path, **detection_config)

inference_time = time.time() - start_time
print(f"Inference completed in {inference_time:.3f} seconds")

# Get results
result = results[0]
print(f"\nDetections found: {len(result.boxes)}")
print(f"Image shape: {result.orig_img.shape}")

In [ ]:
# Download a sample image for testing
import urllib.request
from pathlib import Path

# Create data directory if it doesn't exist
Path('./data').mkdir(exist_ok=True)

# Download a sample image
sample_image_url = "https://ultralytics.com/images/zidane.jpg"
sample_image_path = "./data/sample.jpg"

if not Path(sample_image_path).exists():
    print("Downloading sample image...")
    urllib.request.urlretrieve(sample_image_url, sample_image_path)
    print(f"Sample image saved to {sample_image_path}")
else:
    print(f"Sample image already exists at {sample_image_path}")

## 4. Object Detection on Test Image

First, let's download a sample image for testing:

In [ ]:
# Configure detection parameters
confidence_threshold = 0.45  # Confidence threshold for detections
iou_threshold = 0.45  # IoU threshold for NMS

detection_config = {
    'conf': confidence_threshold,
    'iou': iou_threshold,
    'device': device,
    'max_det': 100,  # Maximum detections
    'classes': None,  # None to detect all classes
}

print("Detection Configuration:")
for key, value in detection_config.items():
    print(f"  {key}: {value}")

## 3. Configuration Settings

In [ ]:
# Load YOLOv8 model
# Options: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (extra-large)

model_size = "yolov8m"  # Change this to use different model sizes
device = 0  # GPU device ID, or 'cpu' for CPU

print(f"Loading YOLOv8 {model_size} model...")
model = YOLO(f"{model_size}.pt")
print(f"Model loaded successfully!")

# Model information
print(f"\nModel Information:")
print(f"Model: {model_size}")
print(f"Device: {device}")
print(f"Task: {model.task}")

## 2. Load Pre-trained YOLOv8 Model

In [ ]:
# Import necessary libraries
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
import pandas as pd
from PIL import Image
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Import Required Libraries

# YOLOv8 Object Detection Project

This notebook demonstrates object detection using YOLOv8 (You Only Look Once v8), a state-of-the-art real-time object detection model.